# 03 – Privacy Demo (Governance Officer)

This notebook is the **Governance Officer** contribution for the project rubric.

It demonstrates, using the *real dataset loaded at runtime*:

1. **PII inventory** (what PII exists and why it matters)
2. **Pseudonymization** (deterministic salted hashing) for at least one PII field
3. **Data minimization** (reduce identifiability + minimize sensitive behavioral detail)
4. **Audit trail** (traceability of governance actions)
5. **GDPR mapping** to specific articles + **EU AI Act** note for credit scoring (academic mapping, not legal advice)

All outputs are written to the `output/` folder at the project root.


## 1) Imports

This notebook uses a small set of Python libraries to keep the privacy workflow easy to run and explain.

- **pandas** is used for tabular analysis and output generation
- **json** supports loading JSON records and writing governance outputs
- **hashlib** is used for deterministic salted hashing in pseudonymization
- **Path** and related utilities make the file loading logic robust across local machines

The aim here is not to overengineer the solution, but to keep the governance controls **clear, reproducible, and easy to defend in viva/Q&A**.


In [1]:
# ------------------------------------------------------------
# Import the libraries needed for the privacy/governance workflow
# ------------------------------------------------------------
# These imports support:
# - dataset loading and flattening
# - pseudonymization via hashing
# - lightweight audit logging
# - writing governance outputs for review
#
# The notebook keeps dependencies simple on purpose so that the
# governance analysis is portable across teammates' computers.

from __future__ import annotations

import ast
import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


## 2) Dataset loading (CSV in `data/` folder)

For this project, the privacy demo runs on the cleaned CSV dataset stored inside the repository:

- `data/cleaned_credit_data.csv`

This keeps the notebook consistent across teammates' machines because everyone can use the same
repository structure instead of editing local file paths.

The code below loads the CSV directly from the repo's `data/` folder and prints the file used
together with the dataset shape for traceability.


In [2]:
# ------------------------------------------------------------
# Step 1: Load the dataset from the repository data/ folder
# ------------------------------------------------------------
# This notebook is designed to read the cleaned CSV produced by the
# data engineering phase from:
#     data/cleaned_credit_data.csv
#
# Keeping the dataset inside the repository makes the notebook easier
# to run on different teammates' computers without changing code.
# ------------------------------------------------------------

# Find the project root by searching upward from current working directory
def find_project_root():
    current = Path.cwd()
    # Search up to 5 levels up for a directory containing both 'data' and 'notebooks' folders
    for _ in range(5):
        if (current / "data").exists() and (current / "notebooks").exists():
            return current
        current = current.parent
    # If not found, try just looking for the data folder
    current = Path.cwd()
    for _ in range(5):
        if (current / "data").exists():
            return current
        current = current.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
CSV_PATH = PROJECT_ROOT / "data" / "cleaned_credit_data.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        "Could not find the dataset at: "
        f"{CSV_PATH.as_posix()}\n"
        f"Current working directory: {Path.cwd().as_posix()}\n"
        f"Project root detected: {PROJECT_ROOT.as_posix()}\n"
        "Make sure 'cleaned_credit_data.csv' is inside the repo's data/ folder."
    )

df = pd.read_csv(CSV_PATH)
source_used = CSV_PATH.as_posix()

print("Loaded source file:", source_used)
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]:,}")
df.head(3)

Loaded source file: c:/Users/divak/Documents/dego-project-team8/dego-project-team8/data/cleaned_credit_data.csv
Rows: 500  Columns: 23


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,notes,ssn_conflict_flag,dob_parsed,age_calculated
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,male,2001-03-09,10036.0,...,31212.0,False,algorithm_risk_score,not_specified,NaN,NaN,NaN,False,2001-09-03,23.3
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,male,1992-03-31,10032.0,...,17915.0,False,algorithm_risk_score,not_specified,NaN,NaN,NaN,False,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,male,1989-10-24,10075.0,...,37909.0,True,NaN,vacation,3.7,59000.0,NaN,False,NaN,NaN


## 3) PII inventory

This section creates a structured **PII inventory**, which is one of the strongest governance artifacts in the notebook.

Instead of only listing column names, the notebook adds:
- a **classification** (direct identifier, quasi-identifier, behavioral data, decision output, etc.)
- a **GDPR-oriented category**
- a **risk level**
- presence/completeness information
- a short **governance note** explaining how the field should be handled

This makes the analysis easier to justify academically because it shows **not only what data exists, but why it matters and what control should be applied**.


In [3]:
# ------------------------------------------------------------
# Step 2: Build a governance-oriented PII inventory
# ------------------------------------------------------------
# A simple list of PII columns is not enough for strong governance work.
# This block creates a richer inventory that documents:
# - what the field is
# - how risky it is
# - how it maps to GDPR/privacy concepts
# - what control should be applied
#
# Including key fields even when they are missing is useful because it
# shows that the governance review is systematic rather than accidental.

# Key fields we must classify even if missing
KEY_FIELDS = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code",
    "applicant_info.gender",
    "spending_behavior",
    "decision.loan_approved",
    "decision.rejection_reason",
    "decision.interest_rate",
    "decision.approved_amount",
    "financials.annual_income",
]

FIELD_POLICY = {
    "applicant_info.full_name": {
        "classification": "Direct identifier",
        "gdpr_category": "Personal data (identifier)",
        "risk_level": "High",
        "governance_note": "Pseudonymize; do not use in modeling; restrict access.",
    },
    "applicant_info.email": {
        "classification": "Direct identifier (contact)",
        "gdpr_category": "Personal data (contact)",
        "risk_level": "High",
        "governance_note": "Pseudonymize; store separately; restrict access.",
    },
    "applicant_info.ssn": {
        "classification": "National identifier",
        "gdpr_category": "Personal data (unique identifier)",
        "risk_level": "High",
        "governance_note": "Strongly protect; pseudonymize; never expose; consider tokenization.",
    },
    "applicant_info.ip_address": {
        "classification": "Online identifier",
        "gdpr_category": "Personal data (online identifier)",
        "risk_level": "High",
        "governance_note": "Pseudonymize; consider truncation; protect in logs.",
    },
    "applicant_info.date_of_birth": {
        "classification": "Quasi-identifier",
        "gdpr_category": "Personal data (demographic)",
        "risk_level": "Medium",
        "governance_note": "Minimize to age band; avoid storing full DOB in operational dataset.",
    },
    "applicant_info.zip_code": {
        "classification": "Quasi-identifier / proxy",
        "gdpr_category": "Personal data (location)",
        "risk_level": "Medium",
        "governance_note": "Minimize to ZIP3 (or similar) to reduce re-identification risk.",
    },
    "applicant_info.gender": {
        "classification": "Protected attribute",
        "gdpr_category": "Personal data (demographic)",
        "risk_level": "Medium",
        "governance_note": "Do not keep in operational dataset; retain only in controlled fairness/audit environment if needed.",
    },
    "spending_behavior": {
        "classification": "Behavioral profiling input",
        "gdpr_category": "Profiling / behavioral data",
        "risk_level": "High",
        "governance_note": "Minimize to aggregates; treat sensitive categories carefully; document purpose/necessity.",
    },
    "decision.loan_approved": {
        "classification": "Automated decision output",
        "gdpr_category": "Decision data (profiling outcome)",
        "risk_level": "High",
        "governance_note": "Maintain audit trail; enable human review; store model version and rationale.",
    },
    "decision.rejection_reason": {
        "classification": "Decision rationale",
        "gdpr_category": "Decision explanation data",
        "risk_level": "High",
        "governance_note": "Maintain traceability; ensure explanations are accurate and reviewable.",
    },
    "decision.interest_rate": {
        "classification": "Decision output",
        "gdpr_category": "Decision data (profiling outcome)",
        "risk_level": "Medium",
        "governance_note": "Maintain traceability and human oversight when algorithmic.",
    },
    "decision.approved_amount": {
        "classification": "Decision output",
        "gdpr_category": "Decision data (profiling outcome)",
        "risk_level": "Medium",
        "governance_note": "Maintain traceability and human oversight when algorithmic.",
    },
    "financials.annual_income": {
        "classification": "Financial attribute",
        "gdpr_category": "Personal data (financial)",
        "risk_level": "Medium",
        "governance_note": "Limit access; use only as necessary; document purpose and retention.",
    },
}

all_columns = sorted(set(df.columns).union(KEY_FIELDS))

rows = []
for col in all_columns:
    present = col in df.columns
    non_null = int(df[col].notna().sum()) if present else 0
    total = int(df.shape[0]) if present else int(df.shape[0])
    non_null_pct = (non_null / total * 100) if total > 0 else 0.0

    policy = FIELD_POLICY.get(col, None)
    if policy:
        classification = policy["classification"]
        gdpr_category = policy["gdpr_category"]
        risk_level = policy["risk_level"]
        note = policy["governance_note"]
    else:
        classification = "Other / business data"
        gdpr_category = "Not classified"
        risk_level = "Low"
        note = "Review if used for profiling or if it can identify a person when combined."

    rows.append({
        "column_name": col,
        "classification": classification,
        "gdpr_category": gdpr_category,
        "risk_level": risk_level,
        "present_in_data": bool(present),
        "non_null_count": non_null,
        "non_null_pct": round(non_null_pct, 2),
        "governance_note": note,
    })

pii_inventory = pd.DataFrame(rows)

# Show key fields first, then the rest
key_order = {name: i for i, name in enumerate(KEY_FIELDS)}
pii_inventory["__sort"] = pii_inventory["column_name"].map(lambda x: key_order.get(x, 10_000))
pii_inventory = pii_inventory.sort_values(["__sort", "risk_level", "column_name"]).drop(columns="__sort")

pii_inventory.head(25)


,column_name,classification,gdpr_category,risk_level,present_in_data,non_null_count,non_null_pct,governance_note
4,applicant_info.full_name,Direct identifier,Personal data (identifier),High,True,500,100.0,Pseudonymize; do not use in modeling; restrict...
3,applicant_info.email,Direct identifier (contact),Personal data (contact),High,True,489,97.8,Pseudonymize; store separately; restrict access.
7,applicant_info.ssn,National identifier,Personal data (unique identifier),High,True,496,99.2,Strongly protect; pseudonymize; never expose; ...
6,applicant_info.ip_address,Online identifier,Personal data (online identifier),High,True,496,99.2,Pseudonymize; consider truncation; protect in ...
2,applicant_info.date_of_birth,Quasi-identifier,Personal data (demographic),Medium,True,495,99.0,Minimize to age band; avoid storing full DOB i...
8,applicant_info.zip_code,Quasi-identifier / proxy,Personal data (location),Medium,True,499,99.8,Minimize to ZIP3 (or similar) to reduce re-ide...
5,applicant_info.gender,Protected attribute,Personal data (demographic),Medium,True,498,99.6,Do not keep in operational dataset; retain onl...
21,spending_behavior,Behavioral profiling input,Profiling / behavioral data,High,True,500,100.0,Minimize to aggregates; treat sensitive catego...
11,decision.loan_approved,Automated decision output,Decision data (profiling outcome),High,True,500,100.0,Maintain audit trail; enable human review; sto...
12,decision.rejection_reason,Decision rationale,Decision explanation data,High,True,208,41.6,Maintain traceability; ensure explanations are...


## 4) Pseudonymization (deterministic salted hashing)

Pseudonymization is used here to reduce direct identification risk while still preserving the ability to link records consistently.

The implementation uses:
- a **salt** from `PSEUDONYM_SALT`
- deterministic **SHA-256 hashing**
- separate `*_pseudo` columns instead of overwriting the originals

That last point is important for explanation: the notebook keeps the original columns in the pseudonymized dataframe only so we can **demonstrate the control clearly**. In a stricter production workflow, raw identifiers would usually be isolated, access-controlled, or removed from downstream datasets.


In [4]:
# ------------------------------------------------------------
# Step 3: Apply pseudonymization to direct identifiers
# ------------------------------------------------------------
# Pseudonymization is used here to reduce the exposure of direct
# identifiers while still preserving consistent record linkage.
#
# Important design choices:
# - A salt is taken from the environment for better practice
# - A visible demo fallback is used only so the notebook remains runnable
# - New *_pseudo columns are created instead of overwriting originals
#
# Keeping original + pseudonymized values side by side in this demo is
# intentional: it makes the transformation easy to inspect and explain.

PSEUDONYM_SALT = os.getenv("PSEUDONYM_SALT")
if not PSEUDONYM_SALT:
    PSEUDONYM_SALT = "DEMO_SALT_CHANGE_ME"
    print("WARNING: using demo salt; set PSEUDONYM_SALT in production")

def salted_hash(value: object) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == "":
        return None
    raw = f"{PSEUDONYM_SALT}::{s}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]

PSEUDO_TARGETS = {
    "applicant_info.ssn": "ssn",
    "applicant_info.email": "email",
    "applicant_info.full_name": "name",
    "applicant_info.ip_address": "ip",
}

df_pseudo = df.copy()
pseudonymized_columns = []

for col, prefix in PSEUDO_TARGETS.items():
    if col in df_pseudo.columns:
        new_col = f"{col}_pseudo"
        if col == "applicant_info.email":
            df_pseudo[new_col] = df_pseudo[col].apply(lambda x: f"{prefix}_{salted_hash(str(x).lower())}")
        else:
            df_pseudo[new_col] = df_pseudo[col].apply(lambda x: f"{prefix}_{salted_hash(x)}")
        pseudonymized_columns.append(new_col)

# Show a small preview (few rows only) of original vs pseudonymized
preview_cols = []
for col in PSEUDO_TARGETS.keys():
    if col in df_pseudo.columns and f"{col}_pseudo" in df_pseudo.columns:
        preview_cols.extend([col, f"{col}_pseudo"])

df_pseudo[preview_cols].head(5)


,applicant_info.ssn,applicant_info.ssn_pseudo,applicant_info.email,applicant_info.email_pseudo,applicant_info.full_name,applicant_info.full_name_pseudo,applicant_info.ip_address,applicant_info.ip_address_pseudo
0,596-64-4340,ssn_523f694b224accec,jerry.smith17@hotmail.com,email_83df9c1280f05445,Jerry Smith,name_cd5f2a132103bd2f,192.168.48.155,ip_7d1c56fc06263ab0
1,425-69-4784,ssn_64495e78c390615a,brandon.walker2@yahoo.com,email_7e7d2f82d5daf109,Brandon Walker,name_745b3671715e7d15,10.1.102.112,ip_bc3012475bfe8b97
2,370-78-5178,ssn_6dd91f788a08a891,scott.moore94@mail.com,email_80a40af78911eac6,Scott Moore,name_88385b38c4a11d64,10.240.193.250,ip_a7ce00da4b477a42
3,194-35-1833,ssn_c0646e01e20a51fc,thomas.lee6@protonmail.com,email_1f69dab7c1a20215,Thomas Lee,name_bb305c6131beb3c7,192.168.175.67,ip_f9ffdad4e62b759f
4,480-41-2475,ssn_a2e206ca874c9e5f,brian.rodriguez86@aol.com,email_35c98d7e7152cdc2,Brian Rodriguez,name_cd032e822bdfcb51,172.29.125.105,ip_0ae9cbb413382b36


## 5) Data minimization

This section applies **data minimization**, which is a core GDPR principle.

The notebook reduces identifiability by:
- generalizing **date of birth** into `age_band`
- generalizing **ZIP code** into `zip3`
- reducing **spending behavior** into a few aggregated indicators

This is important because governance is not only about protecting raw identifiers like SSN or email.  
It is also about reducing the amount of **indirectly identifying or overly detailed personal information** that remains in the operational dataset.

Protected attributes such as `gender` are removed from the normal operational dataset and should only be kept in a more restricted fairness/audit context if truly needed.


In [5]:
# ------------------------------------------------------------
# Step 4: Apply data minimization to the operational dataset
# ------------------------------------------------------------
# This block reduces identifiability and limits unnecessary detail.
#
# Governance logic used here:
# - Replace full DOB with an age band
# - Replace full ZIP code with ZIP3
# - Replace detailed spending records with aggregated indicators
# - Remove direct identifiers and free-text notes from the minimized set
#
# Sensitive/protected attributes such as gender are dropped from the
# operational dataset. If they are needed for fairness analysis, they
# should be handled in a separate controlled audit environment.

SENSITIVE_SPENDING_CATEGORIES = {"Alcohol", "Gambling", "Adult Entertainment"}

def zip_to_zip3(val: object) -> str | None:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    raw = str(val).strip().replace(".0", "")
    digits = re.sub(r"\D", "", raw)
    return digits[:3] if len(digits) >= 3 else None

def age_band_from_dob(dob: object) -> str | None:
    if dob is None or (isinstance(dob, float) and pd.isna(dob)):
        return None
    dt = pd.to_datetime(dob, errors="coerce")
    if pd.isna(dt):
        return None
    today = pd.Timestamp.utcnow().tz_localize(None)
    age = (today - dt).days / 365.25
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    if age < 55:
        return "45-54"
    if age < 65:
        return "55-64"
    return "65+"

def parse_spending(cell: object) -> list[dict]:
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    if isinstance(cell, list):
        return cell
    if isinstance(cell, str):
        try:
            parsed = ast.literal_eval(cell)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

def spending_features(cell: object) -> dict:
    items = parse_spending(cell)
    total = 0.0
    cats = []
    sensitive = False
    for it in items:
        cat = it.get("category")
        amt = it.get("amount", 0)
        if cat:
            cats.append(cat)
            if cat in SENSITIVE_SPENDING_CATEGORIES:
                sensitive = True
        try:
            total += float(amt)
        except Exception:
            pass
    return {
        "spending_total_amount": round(total, 2),
        "spending_category_count": len(cats),
        "has_sensitive_spending": bool(sensitive),
    }

df_min = df_pseudo.copy()

if "applicant_info.date_of_birth" in df_min.columns:
    df_min["age_band"] = df_min["applicant_info.date_of_birth"].apply(age_band_from_dob)

if "applicant_info.zip_code" in df_min.columns:
    df_min["zip3"] = df_min["applicant_info.zip_code"].apply(zip_to_zip3)

if "spending_behavior" in df_min.columns:
    feats = df_min["spending_behavior"].apply(spending_features).apply(pd.Series)
    df_min = pd.concat([df_min, feats], axis=1)

# Drop direct/raw fields from minimized dataset
DROP_COLS = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code",
    "spending_behavior",
    "notes",
    "applicant_info.gender",  # dropped from operational dataset; keep only in controlled fairness/audit environment if needed
]

minimized_dropped = [c for c in DROP_COLS if c in df_min.columns]
df_min = df_min.drop(columns=minimized_dropped)

df_min.head(3)


C:\Users\divak\AppData\Local\Temp\ipykernel_32688\3938903532.py:31: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  today = pd.Timestamp.utcnow().tz_localize(None)


,_id,processing_timestamp,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,...,age_calculated,applicant_info.ssn_pseudo,applicant_info.email_pseudo,applicant_info.full_name_pseudo,applicant_info.ip_address_pseudo,age_band,zip3,spending_total_amount,spending_category_count,has_sensitive_spending
0,app_200,2024-01-15T00:00:00Z,73000.0,23.0,0.20,31212.0,False,algorithm_risk_score,not_specified,NaN,...,23.3,ssn_523f694b224accec,email_83df9c1280f05445,name_cd5f2a132103bd2f,ip_7d1c56fc06263ab0,18-24,100,1517.0,3,True
1,app_037,NaN,78000.0,51.0,0.18,17915.0,False,algorithm_risk_score,not_specified,NaN,...,NaN,ssn_64495e78c390615a,email_7e7d2f82d5daf109,name_745b3671715e7d15,ip_bc3012475bfe8b97,25-34,100,947.0,3,False
2,app_215,NaN,61000.0,41.0,0.21,37909.0,True,NaN,vacation,3.7,...,NaN,ssn_6dd91f788a08a891,email_80a40af78911eac6,name_88385b38c4a11d64,ip_a7ce00da4b477a42,35-44,100,109.0,1,False


## 6) Governance controls summary (for rubric)

This section turns the notebook into something stronger than a one-off technical demo.

It records which controls are:
- **implemented in code**
- **still proposed for policy/process**
- linked to specific **GDPR articles**
- linked to high-level **EU AI Act governance expectations**

This makes it easier to show that the governance work is not limited to coding, but also covers the broader control environment around credit-risk processing.


In [6]:
# ------------------------------------------------------------
# Step 5: Summarise governance controls for documentation
# ------------------------------------------------------------
# This section distinguishes between controls that are implemented in
# code right now and controls that are still governance recommendations.
#
# That distinction is useful in coursework because it shows realistic
# governance thinking: some controls are technical, while others depend
# on policy, legal basis, operations, or organizational process.

governance_controls = [
    {
        "control_name": "PII inventory",
        "purpose": "Identify direct identifiers, quasi-identifiers, profiling inputs, and decision outputs.",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 25", "Art. 5(1)(c)"],
        "related_ai_act_obligation": "Data governance & documentation",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Pseudonymization",
        "purpose": "Reduce re-identification risk by hashing identifiers with a salt.",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 25", "Art. 32"],
        "related_ai_act_obligation": "Record-keeping / traceability support",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Data minimization",
        "purpose": "Reduce granularity (DOB→age band, ZIP→ZIP3) and minimize sensitive behavioral detail.",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 5(1)(c)", "Art. 25"],
        "related_ai_act_obligation": "Data governance",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Audit trail",
        "purpose": "Traceability of processing actions (dataset loaded, transformations applied, outputs written).",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 25", "Art. 32"],
        "related_ai_act_obligation": "Logging / record-keeping",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Consent tracking",
        "purpose": "Capture consent state where applicable; document lawful basis where consent is not used.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 6", "Art. 25"],
        "related_ai_act_obligation": "Governance documentation",
        "owner": "Data Controller / Governance",
    },
    {
        "control_name": "Retention policy",
        "purpose": "Define and enforce retention schedule for raw PII and derived datasets.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 5(1)(e)", "Art. 25"],
        "related_ai_act_obligation": "Record-keeping & lifecycle management",
        "owner": "Governance + Data Engineering",
    },
    {
        "control_name": "Human oversight",
        "purpose": "Human review for automated credit decisions and overrides with documentation.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 22"],
        "related_ai_act_obligation": "Human oversight",
        "owner": "Operations / Risk Team",
    },
    {
        "control_name": "Right-to-erasure workflow",
        "purpose": "Operational process to locate and delete personal data upon request.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 17"],
        "related_ai_act_obligation": "Governance / record-keeping",
        "owner": "Data Controller / Governance",
    },
    {
        "control_name": "Model/version logging",
        "purpose": "Record model version, decision source, and feature snapshot reference for traceability.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 22", "Art. 25"],
        "related_ai_act_obligation": "Logging / traceability",
        "owner": "ML Engineering / Governance",
    },
]

controls_df = pd.DataFrame(governance_controls)
controls_df


,control_name,purpose,implemented_in_code,status,related_gdpr_articles,related_ai_act_obligation,owner
0,PII inventory,"Identify direct identifiers, quasi-identifiers...",True,implemented,"[Art. 25, Art. 5(1)(c)]",Data governance & documentation,Governance Officer
1,Pseudonymization,Reduce re-identification risk by hashing ident...,True,implemented,"[Art. 25, Art. 32]",Record-keeping / traceability support,Governance Officer
2,Data minimization,"Reduce granularity (DOB→age band, ZIP→ZIP3) an...",True,implemented,"[Art. 5(1)(c), Art. 25]",Data governance,Governance Officer
3,Audit trail,Traceability of processing actions (dataset lo...,True,implemented,"[Art. 25, Art. 32]",Logging / record-keeping,Governance Officer
4,Consent tracking,Capture consent state where applicable; docume...,False,proposed,"[Art. 6, Art. 25]",Governance documentation,Data Controller / Governance
5,Retention policy,Define and enforce retention schedule for raw ...,False,proposed,"[Art. 5(1)(e), Art. 25]",Record-keeping & lifecycle management,Governance + Data Engineering
6,Human oversight,Human review for automated credit decisions an...,False,proposed,[Art. 22],Human oversight,Operations / Risk Team
7,Right-to-erasure workflow,Operational process to locate and delete perso...,False,proposed,[Art. 17],Governance / record-keeping,Data Controller / Governance
8,Model/version logging,"Record model version, decision source, and fea...",False,proposed,"[Art. 22, Art. 25]",Logging / traceability,ML Engineering / Governance


## 7) Audit trail 

The audit trail gives the notebook a basic level of **traceability**.

For each important step, the notebook records:
- what action happened
- when it happened
- what dataset was used
- how many records were involved
- useful contextual details

This is valuable because governance controls should be observable and reviewable, not just performed silently.


In [7]:
# ------------------------------------------------------------
# Step 6: Create a lightweight audit trail
# ------------------------------------------------------------
# Governance actions should be traceable. This audit trail records:
# - what action was performed
# - when it happened
# - which dataset was used
# - how many records were involved
#
# In a production system, this would likely be written to a database or
# logging platform, but for the coursework a JSON artifact is enough to
# demonstrate the concept clearly.

RUN_ID = datetime.now(timezone.utc).strftime("privacy_%Y%m%dT%H%M%SZ")

audit_trail = []

def audit(action: str, details: dict) -> None:
    audit_trail.append({
        "run_id": RUN_ID,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "action": action,
        "details": details,
        "dataset_source": CSV_PATH.name,
        "record_count": int(df.shape[0]),
    })

# Events
audit("dataset_loaded", {
    "path": CSV_PATH.as_posix(),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
})

audit("pii_inventory_created", {
    "key_fields_present": [c for c in KEY_FIELDS if c in df.columns],
    "high_risk_fields_present": [c for c in ["spending_behavior", "decision.loan_approved"] if c in df.columns],
})

audit("pseudonymization_applied", {
    "pseudonymized_columns": pseudonymized_columns,
})

audit("data_minimization_applied", {
    "dropped_columns": minimized_dropped,
    "derived_columns": [c for c in ["age_band", "zip3", "spending_total_amount", "spending_category_count", "has_sensitive_spending"] if c in df_min.columns],
})

audit_trail[:3]

[{'run_id': 'privacy_20260304T192355Z',
  'timestamp_utc': '2026-03-04T19:23:55.728966+00:00',
  'action': 'dataset_loaded',
  'details': {'path': 'c:/Users/divak/Documents/dego-project-team8/dego-project-team8/data/cleaned_credit_data.csv',
   'rows': 500,
   'columns': 23},
  'dataset_source': 'cleaned_credit_data.csv',
  'record_count': 500},
 {'run_id': 'privacy_20260304T192355Z',
  'timestamp_utc': '2026-03-04T19:23:55.728966+00:00',
  'action': 'pii_inventory_created',
  'details': {'key_fields_present': ['applicant_info.full_name',
    'applicant_info.email',
    'applicant_info.ssn',
    'applicant_info.ip_address',
    'applicant_info.date_of_birth',
    'applicant_info.zip_code',
    'applicant_info.gender',
    'spending_behavior',
    'decision.loan_approved',
    'decision.rejection_reason',
    'decision.interest_rate',
    'decision.approved_amount',
    'financials.annual_income'],
   'high_risk_fields_present': ['spending_behavior',
    'decision.loan_approved']},
  'd

## 8) Write outputs

This section writes both the required and the enhanced governance outputs to the `output/` folder.

The outputs serve different purposes:
- **PII inventory** supports privacy analysis and documentation
- **pseudonymized data** demonstrates the protection technique
- **minimized data** shows privacy-by-design in practice
- **audit trail** supports traceability
- **governance controls summary** documents the wider control environment
- **privacy governance summary** provides a compact review artifact

Together, these outputs make the notebook easier to assess and easier to explain.


In [8]:
# ------------------------------------------------------------
# Step 7: Write governance outputs to the output/ folder
# ------------------------------------------------------------
# These files act as evidence of the governance work:
# - pii_inventory.csv documents the privacy review
# - credit_data_pseudonymized.csv shows the pseudonymization step
# - credit_data_minimized.csv shows the minimized operational dataset
# - audit_trail.json records traceability information
# - governance_controls_summary.json documents the control environment
# - privacy_governance_summary.csv provides a compact summary for review
#
# Writing these outputs makes the notebook more transparent and easier
# for teammates, markers, and viva examiners to inspect.

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Preserve existing outputs
pii_inventory.to_csv(OUTPUT_DIR / "pii_inventory.csv", index=False)
df_pseudo.to_csv(OUTPUT_DIR / "credit_data_pseudonymized.csv", index=False)
df_min.to_csv(OUTPUT_DIR / "credit_data_minimized.csv", index=False)

with (OUTPUT_DIR / "audit_trail.json").open("w", encoding="utf-8") as f:
    json.dump(audit_trail, f, indent=2, ensure_ascii=False)

# New governance outputs
with (OUTPUT_DIR / "governance_controls_summary.json").open("w", encoding="utf-8") as f:
    json.dump(governance_controls, f, indent=2, ensure_ascii=False)

# A small CSV summary that is easy to show in a viva/Q&A
summary_rows = [
    {"metric": "source_file_used", "value": CSV_PATH.name},
    {"metric": "rows_processed", "value": int(df.shape[0])},
    {"metric": "columns_processed", "value": int(df.shape[1])},
    {"metric": "key_pii_fields_present", "value": ", ".join([c for c in KEY_FIELDS if c in df.columns])},
    {"metric": "pseudonymized_columns", "value": ", ".join(pseudonymized_columns)},
    {"metric": "minimized_columns_dropped", "value": ", ".join(minimized_dropped)},
    {"metric": "outputs_written", "value": ", ".join([
        "pii_inventory.csv",
        "credit_data_pseudonymized.csv",
        "credit_data_minimized.csv",
        "audit_trail.json",
        "governance_controls_summary.json",
        "privacy_governance_summary.csv",
    ])},
]
privacy_governance_summary = pd.DataFrame(summary_rows)
privacy_governance_summary.to_csv(OUTPUT_DIR / "privacy_governance_summary.csv", index=False)

audit("governance_outputs_written", {"output_dir": OUTPUT_DIR.as_posix()})

print("Wrote outputs to:", OUTPUT_DIR.as_posix())
for fname in [
    "pii_inventory.csv",
    "credit_data_pseudonymized.csv",
    "credit_data_minimized.csv",
    "audit_trail.json",
    "governance_controls_summary.json",
    "privacy_governance_summary.csv",
]:
    print("-", (OUTPUT_DIR / fname).as_posix())

Wrote outputs to: c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output
- c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output/pii_inventory.csv
- c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output/credit_data_pseudonymized.csv
- c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output/credit_data_minimized.csv
- c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output/audit_trail.json
- c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output/governance_controls_summary.json
- c:/Users/divak/Documents/dego-project-team8/dego-project-team8/output/privacy_governance_summary.csv


## 9) GDPR mapping (academic) + EU AI Act note

This section maps what we observed and implemented to **specific GDPR articles** (academic mapping, not legal advice):

- **Art. 6 (Lawful basis):** processing must have a lawful basis (e.g., contract, legitimate interest). This must be documented by the controller; code can capture metadata fields (lawful basis label).
- **Art. 5(1)(c) (Data minimization):** we drop direct identifiers (name/email/SSN/IP) from the operational dataset and reduce granularity (DOB→age band, ZIP→ZIP3).
- **Art. 5(1)(e) (Storage limitation):** retention periods should be defined and enforced (e.g., delete raw PII after X days; keep minimized aggregates longer). This notebook recommends policy + metadata fields for retention.
- **Art. 17 (Right to erasure):** an erasure workflow should locate and delete personal data on request. Pseudonymized identifiers help locate records without exposing raw identifiers.
- **Art. 22 (Automated decision-making):** credit approval outcomes and rejection reasons indicate automated decision outputs. Governance should support **human review**, the ability to contest decisions, and documentation of overrides.
- **Art. 25 (Data protection by design and default):** apply privacy controls early (PII inventory, pseudonymization, minimization) and default to minimized operational datasets.
- **Art. 32 (Security of processing):** pseudonymization supports security, but production also requires encryption, access controls, monitoring, and secure key/salt management.

### EU AI Act (academic note)
Credit scoring / creditworthiness assessment is widely treated as a **high-risk AI use case**, so governance expectations include:
- **logging and record-keeping**
- **traceability**
- **human oversight**
- **data governance and quality management**
- **documentation of the system and decisions**

This notebook implements baseline logging/traceability (audit trail + outputs) and recommends governance metadata and oversight processes.
